In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating nowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 70.0 MB/s eta 0:00:00
Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [3]:
from transformers import pipeline

zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
row_150 = train.iloc[150]

prompt_150 = str(row_150["prompt"])

labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

ans_150 = str(row_150[row_150["answer"]])

In [5]:
result = zs(
    prompt_150,
    candidate_labels=labels_150
)

ground_truth_score = result["scores"][
    result["labels"].index(ans_150)
]

print(round(ground_truth_score, 3))

0.384


In [6]:
query_embedding = model.encode([prompt_150])

D, I = index.search(query_embedding.astype(np.float32), k=10)

retrieved = I[0]

if 150 in retrieved:
    rank = np.where(retrieved == 150)[0][0] + 1
    print(rank)
else:
    print("Not found in top 10")

10


In [7]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
docs_10 = [kb[i] for i in retrieved]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)
sorted_indices = np.argsort(ce_scores)[::-1]
reranked_doc_indices = retrieved[sorted_indices]
rank = np.where(reranked_doc_indices == 150)[0][0] + 1

print(rank)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

1


In [8]:
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])
query_embedding = model.encode([prompt_42])
D, I = index.search(query_embedding.astype(np.float32), k=5)
retrieved_indices = I[0]
concatenated_docs = " ".join([kb[i] for i in retrieved_indices])
text = f"Context: {concatenated_docs} Question: {prompt_42}"
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens = tokenizer(text, truncation=False)
print(len(tokens["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

216


In [9]:
true_document = kb[150]
rag_input = f"Context: {true_document} Question: {prompt_150}"
result = zs(
    rag_input,
    candidate_labels=labels_150
)
ground_truth_score = result["scores"][
    result["labels"].index(ans_150)
]

print(round(ground_truth_score, 3))

0.989


In [10]:
wrong_document = kb[999]
adversarial_rag = f"Context: {wrong_document} Question: {prompt_150}"
result = zs(
    adversarial_rag,
    candidate_labels=labels_150
)
ground_truth_score = result["scores"][
    result["labels"].index(ans_150)
]

print(round(ground_truth_score, 3))

0.529


In [11]:
hits = 0

for i in range(100):
    row = train.iloc[i]
    prompt = str(row["prompt"])
    correct_doc = str(row[row["answer"]])
    query_embedding = model.encode([prompt])
    D, I = index.search(query_embedding.astype(np.float32), k=5)
    retrieved_docs = [kb[idx] for idx in I[0]]
    if correct_doc in retrieved_docs:
        hits += 1

hit_rate = (hits / 100) * 100
print(round(hit_rate, 1))

73.0


In [12]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

map3_scores = []

for i in range(20):
    row = train.iloc[i]

    prompt = str(row["prompt"])
    options = [str(row[c]) for c in ["A", "B", "C", "D", "E"]]
    letters = ["A", "B", "C", "D", "E"]
    correct_letter = row["answer"]
    query_embedding = model.encode([prompt]).astype(np.float32)
    D, I = index.search(query_embedding, k=5)
    retrieved_indices = I[0]
    docs = [kb[idx] for idx in retrieved_indices]
    pairs = [[prompt, doc] for doc in docs]
    ce_scores = cross_encoder.predict(pairs)

    best_doc = docs[np.argmax(ce_scores)]
    rag_input = f"Context: {best_doc} Question: {prompt}"
    result = zs(rag_input, candidate_labels=options)

    pred_letters = []
    for label in result["labels"]:
        pred_letters.append(letters[options.index(label)])

    top3 = pred_letters[:3]
    if correct_letter in top3:
        ap = 1 / (top3.index(correct_letter) + 1)
    else:
        ap = 0.0

    map3_scores.append(ap)

print(round(np.mean(map3_scores), 3))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.975
